# Gross-Pitaevskii

The Bose Einstein condensation is descripted by the Gross-Pitaevskii equation. The trap in our case can be very well described by an harmonic oscillator potential.

We assume that all the atoms are in the condensate, $\Psi(\bar{r})$ is normalized: $\int d^3r|\Psi(\bar{r})|^2=1$

In harmonic oscillator units, the GP equation reads:

$[-\frac{1}{2}\bar{\nabla}^2+\frac{1}{2}r^2+4\pi \bar{a}_sN|\Psi(\bar{r})|^2]\Psi(\bar{r}) = \mu\Psi(\bar{r})$


And the energy per particle:

$e = \frac{E}{N}=\int d^3r[-\frac{1}{2}\bar{\nabla}^2+\frac{1}{2}r^2]\Psi(\bar{r})+2\pi \bar{a}_sN\int d^3r|\Psi(\bar{r})|^4$

In a Bose-Einstein condensation all the boson are in the ground state, so, the monoparticular wave function will be:

$\Psi=\frac{R}{r}Y_{00}$ -->normalized to 1

$Y_{00}$ is constant ($Y_{00}=\frac{1}{\sqrt{4\pi}}$) so the equation is:

$[-\frac{1}{2}\frac{d^2}{dr^2}+\frac{1}{2}r^2+4\pi \bar{a}_sN(\frac{R}{r})^2\frac{1}{4\pi}]R(\bar{r}) = \mu R(\bar{r})$

The radial part comes from solving the Schrödinger equation for the ground state with an harmonic potential-->$V(r)=\frac{1}{2}mw^2r^2$, which is needed to trap the paticles to the condensate.
The normalized radial part is:

$R_{00}=2(\frac{m\omega}{\pi\hbar})^{\frac{3}{4}}e^{-\frac{m\omega r^2}{2\hbar}}=2\alpha^{\frac{3}{2}}\pi^{-\frac{1}{4}}e^{-\frac{\alpha^2 r^2}{2}}$

The time operator is:

$U=e^{-\frac{itH}{\hbar}}\xrightarrow[it=\tau-->t = -i\tau]{}U=e^{-\frac{\tau H}{\hbar}}$

We are using imaginary time. The triall wave function that we choose at $\tau=0$ with non-zero overlap with the ground state is:

$\Psi(x,\tau=0)=\sum c_n\psi_n(x)$

Adding the time dependence:

$\Psi(x,\tau)=\sum c_n\psi_n(x)e^{-\frac{E_n\tau}{\hbar}}$

$\tau$ is an imaginary time which if we take it to infinity the wave function results:

$\Psi(x,\tau)= c_0\psi_0(x)e^{-\frac{E_0\tau}{\hbar}}$

The states different than $\phi_0$ decay faster and only survives the ground state which we want to find. 

We will track the evolution with:

$R(r,t+\Delta t)=R(r,t)-\Delta tHR(r,t)$ (essentially a Taylor expansion of the decay $e^{-H\tau}$)

So the idea is to compute the evolution of R(r,t) starting with our trial wave function and updating it with the time step and the eigenvalue of the hamiltonian of R(r,t) (which is the chemical potential). This will lead us to a lower energy wave function, however, we are taking a portion of R(r,t) so we will need to normalized it at each step.

The energy will be:

$e=\int dr[-\frac{1}{2}RR''+\frac{1}{2}R^2r^2]+\frac{\bar{a}_sN}{2}\int drr^2(\frac{R}{r})^4$]

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Math

In [2]:
#hamiltionian for the chemical potential
def H(xr,trailf,step,a0,aa):
    kinetic = -0.5*np.divide(np.gradient(np.gradient(trialf,step),step),trialf,out=np.zeros_like(trialf), where=trialf!=0)
    pot = 0.5*xr**2
    inter = a0*aa*np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**2
    mu = kinetic+pot+inter
    return mu

In [3]:
#energy per particle
def e(trialf,step,xr,a0,aa):
    kinetic = -0.5*trialf*np.gradient(np.gradient(trialf,step),step)
    pot = 0.5*xr**2*trialf**2
    inter = 0.5*a0*aa*xr**2*np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**4
    ene = kinetic+pot+inter
    return np.sum(ene)*step, np.sum(kinetic)*step, np.sum(pot)*step, np.sum(inter)*step #integrated energy

a0) Null interaction term

If we consider the term of interaction a0 equal to zero, GP's equation goes as:

$[-\frac{1}{2}\bar{\nabla}^2+\frac{1}{2}r^2]\Psi(\bar{r}) = \tilde{\mu}\Psi(\bar{r})$

Where we identify the 3D hamiltonian of an harmonic oscillator $H_{HO}=-\frac{1}{2}\bar{\nabla}^2+\frac{1}{2}r^2$. We can note that $\tilde{\mu}$ corresponds to its eigenvalue, which is indeed the monoparticular energy, $\epsilon$.

In [4]:
#aa = 10000 Number of atoms
#n1 number of integration steps
#step integration step
#a0 scattering lenght in h.o. function
#alpha the starting parameter for the h.o. function
#time the step in time
# itera number of iterations
a0,n1,step,aa,time,alpha,itera = tuple(map(float, input("Enter input values separated by space: ").split()))

Enter input values separated by space:  0 600 0.015 10000 0.0001 0.44 50000


In [5]:
#starting wave function

Y00 = np.sqrt(1/(4*np.pi)) #spherical harmonic with l=0
norm = 2*np.sqrt(alpha)**3/np.sqrt(np.sqrt(np.pi)) #starting normalization constant


xr = np.linspace(0, (n1-1)*step, int(n1)) #dimension of the system r
trialf = norm*xr * np.exp(-0.5 * alpha**2 * xr**2) #the value of our trial wave functiuon at each r

In [6]:
as3n=aa*a0*a0*a0 #Interaction strength

In [7]:
#main loop to find the convergence
count = 0

for i in range(int(itera)):
    count += 1

    mu = H(xr,trialf,step,a0,aa)    #chemical potential of trailf
    ene,kinetic,pot,inter = e(trialf,step,xr,a0,aa)   #energy of trailf
    mu[0] = 0                       #contour condition
    
    newf = trialf-time*mu*trialf                #evolution in time of trailf
    newnorm = np.sqrt(np.sum(newf**2) * step)   # new normalization constant

    trialf = newf/newnorm  #update to the new trialf
    
    if count == 200:
        print("ene = ",ene)
        count = 0
        
    if i == itera-1:
        print("ene = ", ene, "mu = ", np.mean(mu)) #we use mean with the chemical potential because it's the same for each r
        latex_str = rf"\epsilon_{{kin}} = {kinetic:.4f}, \quad \epsilon_{{pot}} = {pot:.4f}, \quad \epsilon_{{int}} = {inter:.4f}"
        display(Math(latex_str))

ene =  3.6857661921335776
ene =  3.410555742460917
ene =  3.18124258724615
ene =  2.987799451399555
ene =  2.8229088908846682
ene =  2.6811056821697554
ene =  2.5582252549023625
ene =  2.4510381378574175
ene =  2.3570011750348723
ene =  2.2740842026570296
ene =  2.2006467680107167
ene =  2.135348806024263
ene =  2.077084838146142
ene =  2.0249347690262165
ene =  1.978126592166154
ene =  1.9360077708003034
ene =  1.8980230263065834
ene =  1.8636969194990123
ene =  1.8326200589824773
ene =  1.8044380839384373
ene =  1.7788427903267792
ene =  1.7555649283482144
ene =  1.7343683142577182
ene =  1.7150449841626558
ene =  1.69741118010636
ene =  1.681304005642349
ene =  1.6665786235311444
ene =  1.6531058951784205
ene =  1.6407703821558923
ene =  1.6294686461794894
ene =  1.6191077964126521
ene =  1.6096042427632316
ene =  1.6008826215798238
ene =  1.5928748662986896
ene =  1.585519400501244
ene =  1.5787604347845807
ene =  1.572547352030302
ene =  1.5668341682390357
ene =  1.561579058202732

<IPython.core.display.Math object>

As we can see, once the program is stabilised we get that $\epsilon=\mu=1.50$. We got the same value because we are considering no interaction between particles, hence the monoparticular energy is the same as the energy needed to put in or out a particle in the system, and that is the chemical potential $\mu$.

a) Solving of GP equation for different $N$

From now on we are going to consider different sets of N atoms of $^{87}\mathrm{Rb}$, hence our scattering lenght will be $\tilde{a_s}$ = 0.0043. 

In [8]:
#we added to the main loop to find the convergence a loop to iterate over different numbers of atoms, and we create a table to store the results for each number of atoms. We also save the results in a text file for each number of atoms.
#in order to have a good convergence, we have to adjust the parameters for each number of atoms, so we have defined them as the input file found in campus virtual
a0 = 0.00433
count = 0
aa = (100, 1000, 10000, 100000, 1000000)
table = [] # We create an empty list to store all the results for the table
for j in aa:
    if j == 1000000:
        n1=700
        step=0.015
        time=0.00005
        alpha=0.3
        itera=70000
    elif j == 100000:
        n1=600
        step=0.015
        time=0.0001
        alpha=0.4
        itera=50000
    elif j == 10000:
        n1=400
        step=0.015
        time=0.0001
        alpha=0.8
        itera=40000
    elif j == 1000:
        n1=400
        step=0.02
        time=0.0001
        alpha=0.5
        itera=50000
    else:
        n1=300
        step=0.02
        time=0.0001
        alpha=0.5
        itera=50000   
    
    trialf = norm*xr * np.exp(-0.5 * alpha**2 * xr**2) #for each number of atoms, we reset the trial wave function to the initial one
    as3n=j*a0*a0*a0
    for i in range(int(itera)):
        count += 1

        mu = H(xr,trialf,step,a0,j)    #chemical potential of trailf
        ene, kinetic, pot, inter = e(trialf,step,xr,a0,j)   #energy of trailf
        mu[0] = 0                       #contour condition
        
        newf = trialf-time*mu*trialf                #evolution in time of trailf
        newnorm = np.sqrt(np.sum(newf**2) * step)   # new normalization constant

        trialf = newf/newnorm  #update to the new trialf
        
        if count == 200:
            print(f"N={j} - ene0 = {ene}")
            count = 0
            
        if i == itera-1:
            #we only save the results for the last iteration, which is when we have convergence
            ene_f, kinetic_f, pot_f, inter_f = e(trialf,step,xr,a0,j)

            print("-"*20)
            print(f"---------N={j}---------")
            print(f"N={j} - ene0 = {ene}, mu = {np.mean(mu)}")
            latex_str = rf"N={j} \quad \epsilon_{{kin}} = {kinetic_f:.4f}, \quad \epsilon_{{pot}} = {pot_f:.4f}, \quad \epsilon_{{int}} = {inter_f:.4f}"
            display(Math(latex_str)) 
            table.append({
                'N': j,
                'Pot. Químic (mu)': np.mean(mu),
                'E. Total': ene_f,
                'E. Cinètica': kinetic_f,
                'E. Osc. Harmònic': pot_f,
                'E. Interacció': inter_f,
                'Virial (2K-2V+3I)': 2*kinetic_f - 2*pot_f + 3*inter_f })# We added another element to the table that will be used to check the virial theorem in section d)

            with open(f"gosspi_N_{int(j)}.txt", "w") as f:
                f.write(f"N: {j}\nE_total: {ene}\nmu: {np.mean(mu)}\nE_kin: {kinetic}\nE_pot: {pot}\nE_int: {inter}")

N=100 - ene0 = 2.9199404779439186
N=100 - ene0 = 2.7467742619211974
N=100 - ene0 = 2.597549608441538
N=100 - ene0 = 2.4679299551060887
N=100 - ene0 = 2.354561311799194
N=100 - ene0 = 2.2548084247320905
N=100 - ene0 = 2.1665716581492744
N=100 - ene0 = 2.088157183860517
N=100 - ene0 = 2.0181832119156256
N=100 - ene0 = 1.9555111043122912
N=100 - ene0 = 1.899193994179547
N=100 - ene0 = 1.8484379309649481
N=100 - ene0 = 1.8025721275292206
N=100 - ene0 = 1.761025914402748
N=100 - ene0 = 1.723310700302349
N=100 - ene0 = 1.6890057136097494
N=100 - ene0 = 1.6577466305755855
N=100 - ene0 = 1.6292164297501284
N=100 - ene0 = 1.6031379793395826
N=100 - ene0 = 1.5792679852461025
N=100 - ene0 = 1.5573920161955856
N=100 - ene0 = 1.537320387953045
N=100 - ene0 = 1.5188847376423038
N=100 - ene0 = 1.5019351561522754
N=100 - ene0 = 1.486337774729505
N=100 - ene0 = 1.47197272341717
N=100 - ene0 = 1.4587323956592593
N=100 - ene0 = 1.4465199663520645
N=100 - ene0 = 1.4352481207814147
N=100 - ene0 = 1.4248379

<IPython.core.display.Math object>

N=1000 - ene0 = 3.096552555385583
N=1000 - ene0 = 2.9533985160482916
N=1000 - ene0 = 2.833142120393625
N=1000 - ene0 = 2.7314219776221043
N=1000 - ene0 = 2.6448658490084456
N=1000 - ene0 = 2.570827167698277
N=1000 - ene0 = 2.507201994759831
N=1000 - ene0 = 2.452299021250649
N=1000 - ene0 = 2.4047453969350197
N=1000 - ene0 = 2.363417283859339
N=1000 - ene0 = 2.327387812344858
N=1000 - ene0 = 2.295887508616187
N=1000 - ene0 = 2.268273809990794
N=1000 - ene0 = 2.244007303779153
N=1000 - ene0 = 2.2226330113286146
N=1000 - ene0 = 2.2037655067803117
N=1000 - ene0 = 2.1870769850077822
N=1000 - ene0 = 2.172287622094939
N=1000 - ene0 = 2.1591577352703655
N=1000 - ene0 = 2.1474813676838007
N=1000 - ene0 = 2.1370810103310593
N=1000 - ene0 = 2.1278032379894793
N=1000 - ene0 = 2.1195150845257515
N=1000 - ene0 = 2.11210101976943
N=1000 - ene0 = 2.105460418396956
N=1000 - ene0 = 2.099505433142614
N=1000 - ene0 = 2.0941592017295587
N=1000 - ene0 = 2.0893543303506936
N=1000 - ene0 = 2.0850316071766164


<IPython.core.display.Math object>

N=10000 - ene0 = 7.5558210590632084
N=10000 - ene0 = 6.596945997758947
N=10000 - ene0 = 6.11650796686993
N=10000 - ene0 = 5.829880875957936
N=10000 - ene0 = 5.641743867852081
N=10000 - ene0 = 5.510520668910222
N=10000 - ene0 = 5.415051157746425
N=10000 - ene0 = 5.343404022916396
N=10000 - ene0 = 5.288339740418296
N=10000 - ene0 = 5.245216261151963
N=10000 - ene0 = 5.210925935907958
N=10000 - ene0 = 5.1833150515749065
N=10000 - ene0 = 5.160847980022762
N=10000 - ene0 = 5.142403472850998
N=10000 - ene0 = 5.127146172179013
N=10000 - ene0 = 5.114442841610043
N=10000 - ene0 = 5.103806187122651
N=10000 - ene0 = 5.0948562484016575
N=10000 - ene0 = 5.0872932923160725
N=10000 - ene0 = 5.0808784199652886
N=10000 - ene0 = 5.075419458011593
N=10000 - ene0 = 5.0707605393560735
N=10000 - ene0 = 5.066774303699218
N=10000 - ene0 = 5.063355987202104
N=10000 - ene0 = 5.060418893309022
N=10000 - ene0 = 5.05789088619345
N=10000 - ene0 = 5.055711650168162
N=10000 - ene0 = 5.053830528957019
N=10000 - ene0 =

<IPython.core.display.Math object>

N=100000 - ene0 = 13.13684293812216
N=100000 - ene0 = 12.551201514666191
N=100000 - ene0 = 12.333290832053224
N=100000 - ene0 = 12.23318100910317
N=100000 - ene0 = 12.181398158489518
N=100000 - ene0 = 12.152429457701176
N=100000 - ene0 = 12.135286228790228
N=100000 - ene0 = 12.124703652386925
N=100000 - ene0 = 12.117954939237716
N=100000 - ene0 = 12.113540073661946
N=100000 - ene0 = 12.110593106286045
N=100000 - ene0 = 12.108594051634915
N=100000 - ene0 = 12.107220359209958
N=100000 - ene0 = 12.106266485797473
N=100000 - ene0 = 12.105598485061444
N=100000 - ene0 = 12.105127428261822
N=100000 - ene0 = 12.104793353851234
N=100000 - ene0 = 12.10455530875648
N=100000 - ene0 = 12.104385022196665
N=100000 - ene0 = 12.104262804328654
N=100000 - ene0 = 12.104174840161368
N=100000 - ene0 = 12.104111376946094
N=100000 - ene0 = 12.104065494294828
N=100000 - ene0 = 12.104032260485159
N=100000 - ene0 = 12.104008148278663
N=100000 - ene0 = 12.103990627232653
N=100000 - ene0 = 12.103977877285647
N=10

<IPython.core.display.Math object>

N=1000000 - ene0 = 34.99701616952961
N=1000000 - ene0 = 32.204676568942375
N=1000000 - ene0 = 31.21559260781033
N=1000000 - ene0 = 30.761214033791315
N=1000000 - ene0 = 30.522132340780825
N=1000000 - ene0 = 30.38501861759481
N=1000000 - ene0 = 30.30149987574972
N=1000000 - ene0 = 30.24831135287925
N=1000000 - ene0 = 30.213270178659055
N=1000000 - ene0 = 30.18957106741562
N=1000000 - ene0 = 30.17321318806547
N=1000000 - ene0 = 30.16174424309471
N=1000000 - ene0 = 30.153607936995247
N=1000000 - ene0 = 30.1477871888869
N=1000000 - ene0 = 30.14360053322389
N=1000000 - ene0 = 30.14058160969911
N=1000000 - ene0 = 30.138405518826794
N=1000000 - ene0 = 30.136842520649022
N=1000000 - ene0 = 30.135728168865388
N=1000000 - ene0 = 30.134943592585486
N=1000000 - ene0 = 30.13440219986216
N=1000000 - ene0 = 30.134040539731345
N=1000000 - ene0 = 30.133811915914233
N=1000000 - ene0 = 30.133681858355267
N=1000000 - ene0 = 30.13362487259006
N=1000000 - ene0 = 30.133622082798176
N=1000000 - ene0 = 30.1336

<IPython.core.display.Math object>

In [9]:
#we create a dataframe from the table and we display them
df = pd.DataFrame(table)
col_a = ['N', 'Pot. Químic (mu)', 'E. Total', 'E. Cinètica', 'E. Osc. Harmònic', 'E. Interacció']
display(df[col_a].style.format({
    'N': "{:.0e}", 
    'Pot. Químic (mu)': "{:.5f}",
    'E. Total': "{:.5f}",
    'E. Cinètica': "{:.5f}",
    'E. Osc. Harmònic': "{:.5f}",
    'E. Interacció': "{:.5f}"
}))

,N,Pot. Químic (mu),E. Total,E. Cinètica,E. Osc. Harmònic,E. Interacció
0,1e+02,1.44038,1.29114,0.46761,0.68124,0.14228
1,1e+03,2.62120,2.04341,0.29574,1.16689,0.58078
2,1e+04,6.85434,5.04152,0.24035,2.97683,1.82433
3,1e+05,16.81851,12.10394,0.12367,7.23763,4.74265
4,1e+06,42.86060,30.24209,0.14292,17.46345,12.63572


We can see that the interaction energy grows with the number of particles N, on the other hand, the kinetic energy goes down. This shows the behaviour of the Thomas-Fermi approximation.

b) Thomas-Fermi approximation

The Thomas-Fermi approximation consists in considering the kinetic energy term as negligible. In this approximation, the GP equation goes as:

$[\frac{1}{2}r^2+4\pi \bar{a}_sN(\frac{R}{r})^2\frac{1}{4\pi}]R(\bar{r}) = \mu R(\bar{r})$

We just have to supress the second derivative term from our functions.

In [10]:
#hamiltionian for the chemical potential
def H_tf(xr,trialf,step,a0,aa):
    pot = 0.5*xr**2
    inter = a0*aa*np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**2
    mu = pot+inter
    return mu

In [11]:
#energy per particle
def e_tf(trialf,step,xr,a0,aa):

    pot = 0.5*xr**2*trialf**2
    inter = 0.5*a0*aa*xr**2*np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**4
    ene = pot+inter
    return np.sum(ene)*step, np.sum(pot)*step, np.sum(inter)*step

In [12]:
#main loop to find the convergence
count = 0
aa = (100, 1000, 10000, 100000, 1000000)
table_tf = [] # We create an empty list to store all the results for the table
for j in aa:

    if j == 1000000:
        n1=700
        step=0.015
        time=0.00005
        alpha=0.3
        itera=70000
    elif j == 100000:
        n1=600
        step=0.015
        time=0.0001
        alpha=0.4
        itera=50000
    elif j == 10000:
        n1=400
        step=0.015
        time=0.0001
        alpha=0.8
        itera=40000
    elif j == 1000:
        n1=400
        step=0.02
        time=0.0001
        alpha=0.5
        itera=50000
    else:
        n1=300
        step=0.02
        time=0.0001
        alpha=0.5
        itera=50000   
        
    trialf = norm*xr * np.exp(-0.5 * alpha**2 * xr**2)
    as3n=j*a0*a0*a0
    for i in range(int(itera)):
        count += 1

        mu = H_tf(xr,trialf,step,a0,j)    #chemical potential of trailf
        ene, pot, inter = e_tf(trialf,step,xr,a0,j)   #energy of trailf
        mu[0] = 0                       #contour condition
        
        newf = trialf-time*mu*trialf                #evolution in time of trailf
        newnorm = np.sqrt(np.sum(newf**2) * step)   # new normalization constant

        trialf = newf/newnorm  #update to the new trialf
        
        if count == 200:
            print(f"N={j} - ene0 = {ene}")
            count = 0
            
        if i == itera-1:
            #we only save the results for the last iteration, which is when we have convergence
            ene_f,pot_f, inter_f = e_tf(trialf,step,xr,a0,j)
            print("-"*20)
            print(f"---------N={j}---------")
            print(f"N={j} - ene0 = {ene}, mu = {np.mean(mu)}")
            latex_str = rf"N={j} \quad \epsilon_{{pot}} = {pot_f:.4f}, \quad \epsilon_{{int}} = {inter_f:.4f}"
            display(Math(latex_str)) 
            table_tf.append({
                "N": j,
                'Pot. Químic (mu)': np.mean(mu),
                'E. Total': ene_f,
                'E. Cinètica': 0,
                'E. Osc. Harmònic': pot_f,
                'E. Interacció': inter_f})

            with open(f"gosspi_N_tf_{int(j)}.txt", "w") as f:
                f.write(f"N: {j}\nE_total: {ene}\nmu: {np.mean(mu)}\nE_pot: {pot}\nE_int: {inter}")

N=100 - ene0 = 2.7987653756692357
N=100 - ene0 = 2.610910333164018
N=100 - ene0 = 2.4478593451403636
N=100 - ene0 = 2.3051063534230374
N=100 - ene0 = 2.1791763235621575
N=100 - ene0 = 2.067346059979417
N=100 - ene0 = 1.9674510836769532
N=100 - ene0 = 1.8777491352075337
N=100 - ene0 = 1.7968218349777114
N=100 - ene0 = 1.723502610101897
N=100 - ene0 = 1.656823051540369
N=100 - ene0 = 1.5959724280073915
N=100 - ene0 = 1.5402667399909193
N=100 - ene0 = 1.489124790547453
N=100 - ene0 = 1.442049484545488
N=100 - ene0 = 1.398613070627518
N=100 - ene0 = 1.3584453892318908
N=100 - ene0 = 1.3212244359612717
N=100 - ene0 = 1.2866687251812803
N=100 - ene0 = 1.2545310656516124
N=100 - ene0 = 1.2245934527789764
N=100 - ene0 = 1.196662850640248
N=100 - ene0 = 1.170567688086861
N=100 - ene0 = 1.1461549317783817
N=100 - ene0 = 1.1232876282744102
N=100 - ene0 = 1.101842829745617
N=100 - ene0 = 1.081709835182303
N=100 - ene0 = 1.0627886924462073
N=100 - ene0 = 1.044988917056635
N=100 - ene0 = 1.028228391

<IPython.core.display.Math object>

N=1000 - ene0 = 2.9767632432458693
N=1000 - ene0 = 2.82064148851303
N=1000 - ene0 = 2.688601756745049
N=1000 - ene0 = 2.5761003991958247
N=1000 - ene0 = 2.479626914649939
N=1000 - ene0 = 2.396426284348489
N=1000 - ene0 = 2.3243069547631174
N=1000 - ene0 = 2.261505046886098
N=1000 - ene0 = 2.2065863708004345
N=1000 - ene0 = 2.158374414528597
N=1000 - ene0 = 2.11589653355958
N=1000 - ene0 = 2.0783431271464035
N=1000 - ene0 = 2.0450362379542
N=1000 - ene0 = 2.015405097008809
N=1000 - ene0 = 1.988966862511346
N=1000 - ene0 = 1.9653112955862801
N=1000 - ene0 = 1.944088457758488
N=1000 - ene0 = 1.9249987545452885
N=1000 - ene0 = 1.9077848198642295
N=1000 - ene0 = 1.892224858646653
N=1000 - ene0 = 1.8781271545716018
N=1000 - ene0 = 1.8653255159767117
N=1000 - ene0 = 1.853675482458658
N=1000 - ene0 = 1.8430511520818798
N=1000 - ene0 = 1.8333425177165696
N=1000 - ene0 = 1.824453223122405
N=1000 - ene0 = 1.8162986666269911
N=1000 - ene0 = 1.8088043938061038
N=1000 - ene0 = 1.801904731326036
N=10

<IPython.core.display.Math object>

N=10000 - ene0 = 7.203134103605994
N=10000 - ene0 = 6.281078399469358
N=10000 - ene0 = 5.819872254242766
N=10000 - ene0 = 5.545348502840488
N=10000 - ene0 = 5.365487981659801
N=10000 - ene0 = 5.24016495738888
N=10000 - ene0 = 5.148987080170809
N=10000 - ene0 = 5.080480997304377
N=10000 - ene0 = 5.027703721131071
N=10000 - ene0 = 4.986216873243754
N=10000 - ene0 = 4.95305893694296
N=10000 - ene0 = 4.926184584618104
N=10000 - ene0 = 4.904140670177173
N=10000 - ene0 = 4.88586999841091
N=10000 - ene0 = 4.870587762963822
N=10000 - ene0 = 4.857701144434687
N=10000 - ene0 = 4.84675550274697
N=10000 - ene0 = 4.837397481840113
N=10000 - ene0 = 4.829349168296376
N=10000 - ene0 = 4.82238965035905
N=10000 - ene0 = 4.816341637528759
N=10000 - ene0 = 4.811061606616159
N=10000 - ene0 = 4.8064324470744415
N=10000 - ene0 = 4.8023579047996074
N=10000 - ene0 = 4.798758338068491
N=10000 - ene0 = 4.795567442888759
N=10000 - ene0 = 4.7927297028237215
N=10000 - ene0 = 4.790198385980343
N=10000 - ene0 = 4.787

<IPython.core.display.Math object>

N=100000 - ene0 = 13.037297698238481
N=100000 - ene0 = 12.450652729852639
N=100000 - ene0 = 12.229499873477357
N=100000 - ene0 = 12.125886335105216
N=100000 - ene0 = 12.070778351117859
N=100000 - ene0 = 12.038774741583794
N=100000 - ene0 = 12.018903973391291
N=100000 - ene0 = 12.005887591041772
N=100000 - ene0 = 11.996975166474856
N=100000 - ene0 = 11.990640591316268
N=100000 - ene0 = 11.985992280589848
N=100000 - ene0 = 11.98248620096749
N=100000 - ene0 = 11.979777743179676
N=100000 - ene0 = 11.977641383049066
N=100000 - ene0 = 11.975925221930275
N=100000 - ene0 = 11.974524296869879
N=100000 - ene0 = 11.97336439304725
N=100000 - ene0 = 11.972391937930022
N=100000 - ene0 = 11.97156752650715
N=100000 - ene0 = 11.970861674653513
N=100000 - ene0 = 11.970251973972506
N=100000 - ene0 = 11.969721148184414
N=100000 - ene0 = 11.969255701484354
N=100000 - ene0 = 11.968844962985214
N=100000 - ene0 = 11.968480400819438
N=100000 - ene0 = 11.968155122799612
N=100000 - ene0 = 11.967863508086076
N=10

<IPython.core.display.Math object>

N=1000000 - ene0 = 34.9491933169014
N=1000000 - ene0 = 32.15743763506541
N=1000000 - ene0 = 31.16715468607383
N=1000000 - ene0 = 30.711064650285685
N=1000000 - ene0 = 30.470106600823748
N=1000000 - ene0 = 30.331084434559642
N=1000000 - ene0 = 30.245683730491432
N=1000000 - ene0 = 30.190667505993993
N=1000000 - ene0 = 30.15386680606906
N=1000000 - ene0 = 30.1284838413906
N=1000000 - ene0 = 30.110522278333374
N=1000000 - ene0 = 30.097532838598184
N=1000000 - ene0 = 30.08796135208424
N=1000000 - ene0 = 30.080792104900365
N=1000000 - ene0 = 30.07534418972294
N=1000000 - ene0 = 30.07115090380139
N=1000000 - ene0 = 30.067886045184963
N=1000000 - ene0 = 30.065317615026647
N=1000000 - ene0 = 30.063278021598503
N=1000000 - ene0 = 30.061644486202763
N=1000000 - ene0 = 30.06032590629989
N=1000000 - ene0 = 30.059253892809117
N=1000000 - ene0 = 30.0583765576193
N=1000000 - ene0 = 30.057654144717215
N=1000000 - ene0 = 30.05705591685315
N=1000000 - ene0 = 30.05655790971109
N=1000000 - ene0 = 30.05614

<IPython.core.display.Math object>

In [14]:
#we create a dataframe from the table and we display them
dg = pd.DataFrame(table_tf)

display(dg.style.format({
    'N': "{:.0e}", 
    'Pot. Químic (mu)': "{:.5f}",
    'E. Total': "{:.5f}",
    'E. Osc. Harmònic': "{:.5f}",
    'E. Interacció': "{:.5f}"
}))

,N,Pot. Químic (mu),E. Total,E. Cinètica,E. Osc. Harmònic,E. Interacció
0,1e+02,13.55956,0.67723,0,0.42491,0.25232
1,1e+03,13.84406,1.69101,0,1.02433,0.66668
2,1e+04,15.26516,4.76301,0,2.85560,1.90741
3,1e+05,20.63247,11.96395,0,7.18029,4.78366
4,1e+06,42.05284,30.05362,0,17.98420,12.06942


c) $\rho(r_1)$ plot

we will make a plot for $\int dr_1r_1^2\rho(r1)=1$

The density is defined as: $\rho(r_1) = |\Psi|^2=|\frac{R(r)}{r}|^2$

Given the definition above, the $r_1^2$ cancels and lasts the squared modulus of the radial part of the wave function integrated by all r, which is ineed 1.

For N = 1000 and N = 100000 with and without TF approximation.

TF approximation:

In [15]:
aa = (1000, 100000)
densities_tf = []
for j in aa:
    if j == 100000:
        n1=600
        step=0.015
        time=0.0001
        alpha=0.4
        itera=50000

    else:
        n1=400
        step=0.02
        time=0.0001
        alpha=0.5
        itera=50000

    trialf = norm*xr * np.exp(-0.5 * alpha**2 * xr**2)
    as3n=j*a0*a0*a0
    for i in range(int(itera)):
        count += 1

        mu = H_tf(xr,trialf,step,a0,j)    #chemical potential of trailf
        ene, pot, inter = e_tf(trialf,step,xr,a0,j)   #energy of trailf
        mu[0] = 0                       #contour condition
        
        newf = trialf-time*mu*trialf                #evolution in time of trailf
        newnorm = np.sqrt(np.sum(newf**2) * step)   # new normalization constant

        trialf = newf/newnorm  #update to the new trialf
    densities_tf.append(np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**2) #density

Without approximation:

In [ ]:
aa = (1000, 100000)
densities = []
for j in aa:
    if j == 100000:
        n1=600
        step=0.015
        time=0.0001
        alpha=0.4
        itera=50000

    else:
        n1=400
        step=0.02
        time=0.0001
        alpha=0.5
        itera=50000
    trialf = norm*xr * np.exp(-0.5 * alpha**2 * xr**2)
    as3n=j*a0*a0*a0
    for i in range(int(itera)):
        count += 1

        mu = H(xr,trialf,step,a0,j)    #chemical potential of trailf
        ene,kinetic, pot, inter = e(trialf,step,xr,a0,j)   #energy of trailf
        mu[0] = 0                       #contour condition
        
        newf = trialf-time*mu*trialf                #evolution in time of trailf
        newnorm = np.sqrt(np.sum(newf**2) * step)   # new normalization constant

        trialf = newf/newnorm  #update to the new trialf
    densities.append(np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**2) #density

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))


N_labels = [1000, 100000]

for i in range(len(N_labels)):
    # We plot the density obtained from the Gross-Pitaevskii equation without approximation
    axes[i].plot(xr[1:], densities[i][1:], label=f"N = {N_labels[i]} GP (no approx)", 
                 color='blue', linewidth=2, alpha=0.7)
    
    # We plot the Thomas-Fermi (TF) approximation
    axes[i].plot(xr[1:], densities_tf[i][1:], '--', label=f"N = {N_labels[i]} TF approx", 
                 color='red', linewidth=2)
    
    axes[i].set_xlabel(r"$r_1$")
    axes[i].set_ylabel(r"$\rho(r_1)$")
    axes[i].set_title(f"Density profiles for N = {N_labels[i]}")
    axes[i].legend()
    axes[i].grid(True, linestyle=':', alpha=0.6)
    
    # We adjust the x-axis limit for each N value
    if N_labels[i] == 1000:
        axes[i].set_xlim(0, 6)
    else:
        axes[i].set_xlim(0, 10)

plt.tight_layout()
plt.show()

We notice that as $N$ increases, the Thomas-Fermi approximation becomes more reliable, while for $N=1000$ it fails to adjust to GP between $r_1 = 0$ and $r_1=3$.

d) Verification of the Virial Theorem for different values of N

The Virial theorem applied to the GP's equation takes the form of:

$2\epsilon_{kin}-2\epsilon_{ho}+3\epsilon_{int}=0$

In section a) we have already calculated this relation. So we only need to display the values in a table:

In [40]:
cols_virial = ['N', 'Virial (2K-2V+3I)']

display(df[cols_virial].style.format({
    'N': "{:.0e}", 
    'Virial (2K-2V+3I)': "{:.2e}" 
}))

,N,Virial (2K-2V+3I)
0,1e+02,3.17e-04
1,1e+03,1.70e-04
2,1e+04,8.38e-05
3,1e+05,3.73e-05
4,1e+06,1.81e-05


As we can see, the sums for each value of N are approximately 0, reaching values of the order of $10^{-4}$ or $10^{-5}$. Hence, we can affirm that the GP's equation fulfills the Virial theorem for all the N values studied.